# Data Preprocessing
## Objectives
- label encoding - convert to numerical categories
- symmetry handling - ensure dataset contains equally as many rows where player A wins as where player B wins - this is achievable by duplicating each row, but swapping the A-B metrics.
- prevent data leakage - Avoid future stats used as features
## Key derived metrics
- Elo rating - relative skill level calculation
- breakpoint conversion - ability to capitalise on high leverage
- service points win % - ability to consistently win when provided an advantage
- return points win % - represents defensive ability
- surface-specific win % - different surfaces play differently - filtered specifically for the current match surface
- win % over past few matches - useful for capturing momentum/tiredness


>! NOTE - This sketched-out notebook has been transferred into a separate data_processing module.

In [154]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [155]:

matches = pd.concat([pd.read_csv(f'../tennis_atp/atp_matches_{year}.csv') for year in range(2020, 2025)])

matches.columns = [
    "tournament_id",
    "tournament_name",
    "surface",
    "draw_size",
    "tournament_level",
    "tournament_date",
    "match_number",
    "winner_id",
    "winner_seed",
    "winner_entry_type",
    "winner_name",
    "winner_hand",
    "winner_height_cm",
    "winner_country_code",
    "winner_age",
    "loser_id",
    "loser_seed",
    "loser_entry_type",
    "loser_name",
    "loser_hand",
    "loser_height_cm",
    "loser_country_code",
    "loser_age",
    
    # Final match score
    "score",

    # Best of how many sets (usually 3/4/5)
    "best_of_sets",

    # Which round of tournament? (F, QF, Q1, Q2, R16, R32)
    "round",

    # How long was the math?
    "match_duration_minutes",
    
    # Winner/loser match stats
    "winner_aces",
    "winner_double_faults",
    "winner_service_points_played",
    "winner_first_serves_in",
    "winner_first_serve_points_won",
    "winner_second_serve_points_won",
    "winner_service_games_played",
    "winner_break_points_saved",
    "winner_break_points_faced",

    "loser_aces",
    "loser_double_faults",
    "loser_service_points_played",
    "loser_first_serves_in",
    "loser_first_serve_points_won",
    "loser_second_serve_points_won",
    "loser_service_games_played",
    "loser_break_points_saved",
    "loser_break_points_faced",
    
    # Rank and rank points
    "winner_rank",
    "winner_rank_points",
    "loser_rank",
    "loser_rank_points"
]

matches['tournament_date'] = pd.to_datetime(matches['tournament_date'], format='%Y%m%d')

In [156]:
matches['unique_match_id'] = matches['tournament_id'] + '_' + matches['match_number'].astype(str)

# Breakpoint conversion stats - these depend on the opposite player's metrics, so have to be added earlier in the data pipeline
matches['winner_bp_conversion'] = (matches['loser_break_points_faced'] - matches['loser_break_points_saved']) / matches['loser_break_points_faced']
matches['loser_bp_conversion'] = (matches['winner_break_points_faced'] - matches['winner_break_points_saved']) / matches['winner_break_points_faced']

In [158]:
# The 'wide data format' - per-match, cana be converted to
# 'long data format', where each player performance in each match is a separate row
# This allows for efficient computation of rolling averages.
def convert_matches_to_long_data(matches):
    neutral_columns = ['unique_match_id', 'tournament_date', 'surface', 'tournament_level', 'best_of_sets']
    
    winner_columns = [column for column in matches.columns if 'winner' in column] + neutral_columns
    winners = matches[winner_columns].copy()
    winners.columns = [column.replace('winner_', '') for column in winners.columns]
    winners['outcome'] = 1

    loser_columns = [column for column in matches.columns if 'loser' in column] + neutral_columns
    losers = matches[loser_columns].copy()
    losers.columns = [column.replace('loser_', '') for column in losers.columns]
    losers['outcome'] = 0

    return pd.concat([winners, losers]).sort_values(['tournament_date', 'unique_match_id'])

# Generate long data
long_data = convert_matches_to_long_data(matches)

long_data.columns

Index(['id', 'seed', 'entry_type', 'name', 'hand', 'height_cm', 'country_code',
       'age', 'aces', 'double_faults', 'service_points_played',
       'first_serves_in', 'first_serve_points_won', 'second_serve_points_won',
       'service_games_played', 'break_points_saved', 'break_points_faced',
       'rank', 'rank_points', 'bp_conversion', 'elo', 'unique_match_id',
       'tournament_date', 'surface', 'tournament_level', 'best_of_sets',
       'outcome'],
      dtype='str')

In [159]:
# --- 1. SERVICE METRICS ---
long_data['serve_win_ratio'] = (
    (long_data['first_serve_points_won'] + long_data['second_serve_points_won']) / 
    long_data['service_points_played']
)
long_data['first_serve_ratio'] = long_data['first_serves_in'] / long_data['service_points_played']
long_data['first_serve_win_ratio'] = long_data['first_serve_points_won'] / long_data['first_serves_in']
long_data['second_serve_win_ratio'] = (
    long_data['second_serve_points_won'] / 
    (long_data['service_points_played'] - long_data['first_serves_in'] - long_data['double_faults'])
)
long_data['ace_rate'] = long_data['aces'] / long_data['service_points_played']
long_data['double_fault_rate'] = long_data['double_faults'] / long_data['service_points_played']

# --- 2. BREAK POINT METRICS ---
long_data['bp_save_rate'] = long_data['break_points_saved'] / long_data['break_points_faced']
long_data['bp_faced_per_service_game'] = long_data['break_points_faced'] / long_data['service_games_played']


# --- 3. ROLLING AVERAGES ---
def create_rolling_stat(df, group_col, stat_col, window=10, min_periods=2):
    return df.groupby(group_col)[stat_col] \
        .transform(lambda x: x.shift(1).rolling(window, min_periods=min_periods).mean())

long_data['rolling_serve_win_ratio'] = create_rolling_stat(long_data, 'id', 'serve_win_ratio')
long_data['rolling_first_serve_ratio'] = create_rolling_stat(long_data, 'id', 'first_serve_ratio')
long_data['rolling_first_serve_win_ratio'] = create_rolling_stat(long_data, 'id', 'first_serve_win_ratio')
long_data['rolling_bp_conversion'] = create_rolling_stat(long_data, 'id', 'bp_conversion')
long_data['rolling_bp_save_rate'] = create_rolling_stat(long_data, 'id', 'bp_save_rate')
long_data['rolling_surface_win_ratio'] = create_rolling_stat(long_data, ['id', 'surface'], 'outcome')
long_data['rolling_ace_rate'] = create_rolling_stat(long_data, 'id', 'ace_rate')
long_data['rolling_double_fault_rate'] = create_rolling_stat(long_data, 'id', 'double_fault_rate')

# --- 4. RECENT FORM & MOMENTUM ---
long_data['recent_form_ema'] = long_data.groupby('id')['outcome'] \
    .transform(lambda x: x.shift(1).ewm(span=5, min_periods=3).mean())

long_data['win_streak'] = long_data.groupby('id')['outcome'] \
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).sum())

# --- 5. FATIGUE & WORKLOAD ---
long_data['days_since_last_match'] = long_data.groupby('id')['tournament_date'] \
    .diff().dt.days.fillna(14)

def get_time_based_feature(df, group_col, date_col, feature_col, days_back):
    """
    Get a feature value from N days ago for each row.
    
    Parameters:
    -----------
    df : DataFrame
    group_col : str - grouping column (e.g., 'id' for player)
    date_col : str - date column name
    feature_col : str - feature to look back (e.g., 'rank')
    days_back : int - number of days to look back
    
    Returns:
    --------
    Series with historical feature values
    """
    result = []
    
    for idx, row in df.iterrows():
        grp = row[group_col]
        current_date = row[date_col]
        cutoff_date = current_date - pd.Timedelta(days=days_back)
        
        # Get historical data for this group before cutoff
        mask = (df[group_col] == grp) & (df[date_col] <= cutoff_date)
        historical = df.loc[mask, feature_col]
        
        if len(historical) > 0:
            result.append(historical.iloc[-1])
        else:
            result.append(np.nan)
    
    return pd.Series(result, index=df.index)


# Usage:
long_data['rank_30d_ago'] = get_time_based_feature(
    long_data, 
    group_col='id',
    date_col='tournament_date',
    feature_col='rank',
    days_back=30
)

long_data['rank_change_30d'] = long_data['rank_30d_ago'] - long_data['rank']

# --- 7. SEEDING ---
long_data['has_seed'] = long_data['seed'].notna().astype(int)
long_data = long_data.drop('seed', axis=1)

In [160]:
# the numerical stats which have 'nan' values to correct by averaging
numeric_nan_columns = [
    'ace_rate', 
    'aces', 
    'age', 
    'bp_conversion', 'bp_faced_per_service_game', 'bp_save_rate', 
    'break_points_faced', 'break_points_saved', 
    'double_fault_rate', 'double_faults', 
    'first_serve_points_won', 'first_serve_ratio', 'first_serve_win_ratio', 'first_serves_in', 
    'height_cm', 
    'rank', 'rank_30d_ago', 'rank_change_30d', 'rank_points', 
    'recent_form_ema', 
    'rolling_bp_conversion', 'rolling_bp_save_rate', 'rolling_first_serve_ratio', 
    'rolling_first_serve_win_ratio', 'rolling_serve_win_ratio', 'rolling_surface_win_ratio',
    'rolling_ace_rate', 'rolling_double_fault_rate',
    'second_serve_points_won', 'second_serve_win_ratio', 
    'serve_win_ratio', 'service_games_played', 'service_points_played', 
    'win_streak'
]

long_data[numeric_nan_columns] = long_data[numeric_nan_columns].fillna(long_data[numeric_nan_columns].mean())

## Label Encoding

In [161]:

# This dictionary is used to map string categories to an integer
entry_rank_map = {
    np.nan: 0,   
    'SE': 1,     # Special Exempt (High Momentum)
    'PR': 2,     # Protected Ranking
    'UP': 2,     # Used Protection (similar to PR)
    'Q': 3,      # Qualifier
    'WC': 4,     # Wild Card
    'LL': 5,     # Lucky Loser
    'Alt': 6,    # Alternate
    'ALT': 6,    # Alternate (capitalisation variant)
    'ITF': 6,    # ITF World Tennis Tour entry
    'W': 6       # Winner placeholder/lower-tier entry
}

long_data['entry_type'] = long_data['entry_type'].map(entry_rank_map)


long_data['hand'] = (long_data['hand'] == 'R').astype(int)

surface_rank_map = {
    np.nan: 0,
    'Hard': 1, # Hard 
    'Clay': 2,
    'Grass': 3
}
long_data['surface'] = long_data['surface'].map(surface_rank_map)\

tournament_level_rank_map = {
    'G': 0, 
    'F': 1, 
    'M': 2, 
    'A': 3, 
    'O': 4, 
    'D': 5
}
long_data['tournament_level'] = long_data['tournament_level'].map(tournament_level_rank_map)

In [162]:
def convert_long_to_symmetric_wide(long_data):
    neutral_columns = ['unique_match_id', 'tournament_date', 'surface', 'tournament_level', 'best_of_sets']
    player_columns = [col for col in long_data.columns if col not in neutral_columns and col not in ['id', 'country_code']]
    winners = long_data[long_data['outcome'] == 1]
    losers = long_data[long_data['outcome'] == 0]

    v1_winners = winners.rename(columns={col: 'A_' + col for col in player_columns})
    v1_losers = losers.rename(columns={col: 'B_' + col for col in player_columns})


    wide_v1 = pd.merge(
        v1_winners[neutral_columns + ['A_' + col for col in player_columns]],
        v1_losers[['unique_match_id'] + ['B_' + col for col in player_columns]],
        on='unique_match_id'
    )


    wide_v1['target'] = 1
    v2_losers = losers.rename(columns={col: 'A_' + col for col in player_columns})
    v2_winners = winners.rename(columns={col: 'B_' + col for col in player_columns})
    
    wide_v2 = pd.merge(
        v2_losers[neutral_columns + ['A_' + col for col in player_columns]],
        v2_winners[['unique_match_id'] + ['B_' + col for col in player_columns]],
        on='unique_match_id'
    )
    wide_v2['target'] = 0

    print('A_rank_change_30d' in wide_v1)
    print('A_rank_change_30d' in wide_v2)

    return pd.concat([wide_v1, wide_v2], axis=0).reset_index(drop=False)

wide_data = convert_long_to_symmetric_wide(long_data)


True
True


In [163]:
# For Elo, Rank, Rank Points, Rolling Serve Win Ratio, Rolling Surface Win Ratio,
# Also physical attributes like Age, and Height
# We convert to differentials (rather than absolute values), as the actual values in isolation are unimportant:

differential_columns = [
    'rank',
    'rank_points',
    'elo',
    'age',
    'height_cm',
    'serve_win_ratio',
    'first_serve_ratio',
    'first_serve_win_ratio',
    'second_serve_win_ratio',
    'ace_rate',
    'double_fault_rate',
    'bp_conversion',
    'bp_save_rate',
    'bp_faced_per_service_game',
    'rolling_serve_win_ratio',
    'rolling_first_serve_ratio',
    'rolling_first_serve_win_ratio',
    'rolling_bp_conversion',
    'rolling_bp_save_rate',
    'rolling_surface_win_ratio',
    'rolling_ace_rate',
    'rolling_double_fault_rate',
    'recent_form_ema',
    'win_streak',
    'rank_change_30d',
    'days_since_last_match',
]

for column in differential_columns:
    wide_data[f'{column}_diff'] = wide_data[f'A_{column}'] - wide_data[f'B_{column}']

    # Drop the original columns
    wide_data = wide_data.drop(f'A_{column}', axis=1)
    wide_data = wide_data.drop(f'B_{column}', axis=1)

wide_data

,index,unique_match_id,tournament_date,surface,tournament_level,best_of_sets,A_entry_type,A_name,A_hand,A_aces,...,rolling_first_serve_win_ratio_diff,rolling_bp_conversion_diff,rolling_bp_save_rate_diff,rolling_surface_win_ratio_diff,rolling_ace_rate_diff,rolling_double_fault_rate_diff,recent_form_ema_diff,win_streak_diff,rank_change_30d_diff,days_since_last_match_diff
0,0,2020-0451_271,2020-01-06,1,3,3,0,Mikhail Kukushkin,1,7.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
1,1,2020-0451_272,2020-01-06,1,3,3,0,Pierre Hugues Herbert,1,6.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
2,2,2020-0451_273,2020-01-06,1,3,3,0,Laslo Djere,1,5.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
3,3,2020-0451_275,2020-01-06,1,3,3,0,Miomir Kecmanovic,1,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
4,4,2020-0451_276,2020-01-06,1,3,3,4,Cem Ilkel,1,11.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26343,13169,2024-7696_396,2024-12-18,1,1,5,0,Arthur Fils,1,9.0,...,0.210740,-0.024443,0.056458,0.155556,0.094811,0.019131,-0.121230,2.0,-69.0,0.0
26344,13170,2024-7696_397,2024-12-18,1,1,5,0,Jakub Mensik,1,2.0,...,-0.034139,-0.069388,-0.086942,0.100000,0.028483,-0.009183,-0.000983,1.0,3.0,0.0
26345,13171,2024-7696_398,2024-12-18,1,1,5,0,Alex Michelsen,1,7.0,...,0.094672,-0.059166,-0.025999,0.200000,0.045745,0.006699,0.203858,2.0,-66.0,0.0
26346,13172,2024-7696_399,2024-12-18,1,1,5,0,Luca Van Assche,1,7.0,...,-0.036621,0.008112,-0.118767,-0.433333,-0.015921,0.040399,-0.489013,-3.0,-29.0,0.0


# Feature Extraction and Serialisation

In [167]:
feature_columns = [
    'A_entry_type',
    'A_hand',
    'A_has_seed',
    'B_entry_type',
    'B_hand',
    'B_has_seed',
    'surface',
    'tournament_level',
    'best_of_sets',
    'rank_diff',
    'rank_points_diff',
    'elo_diff',
    'age_diff',
    'height_cm_diff',
    'rolling_serve_win_ratio_diff',
    'rolling_first_serve_ratio_diff',
    'rolling_first_serve_win_ratio_diff',
    'rolling_bp_conversion_diff',
    'rolling_bp_save_rate_diff',
    'rolling_surface_win_ratio_diff',
    'rolling_ace_rate_diff',
    'rolling_double_fault_rate_diff',
    'recent_form_ema_diff',
    'win_streak_diff',
    'rank_change_30d_diff',
    'days_since_last_match_diff'
]

x = wide_data[feature_columns]
y = wide_data['target']

x.to_csv('../processed_data/tennis_model_features.csv', index=False)
y.to_csv('../processed_data/tennis_model_labels.csv', index=False)

In [ ]:
x.loc[5000]